# Конспект. Модуль 4: Коллаборативная фильтрация — Item-Based (IB-CF)

**Курс:** Мини-курс RecSys (13 модулей)
**Модуль:** 4 из 13 — «Коллаборативная фильтрация: Item-Based (IB-CF)»
**Цель модуля:** решить структурную проблему предыдущего модуля (вычислительная сложность и нестабильность UB-CF, разобранные в 3.3.1–3.3.2) за счёт смены объекта сравнения — с пользователей на товары. Для прямого сравнения результатов мы используем **ту же самую** матрицу оценок, что и в Модуле 3, и предсказываем **тот же самый** недостающий рейтинг — это позволит увидеть на практике, что два разных алгоритма могут давать разные ответы на один и тот же вопрос, и понять, почему.

**Историческая справка:** Item-Based CF был формализован в статье Sarwar et al., «Item-Based Collaborative Filtering Recommendation Algorithms» (2001), а по-настоящему стал известен благодаря статье инженеров Amazon — Linden, Smith, York, «Amazon.com Recommendations: Item-to-Item Collaborative Filtering» (2003), где было показано, что подход отлично масштабируется на каталог из миллионов товаров — именно на этой идее долгое время строился знаменитый блок Amazon «Customers who bought this item also bought».

## 4.1 Идея и ключевой инсайт

### 4.1.1 Аналогия и переформулировка

Модуль 3 отвечал на вопрос «на кого похож этот пользователь?». Модуль 4 задаёт принципиально другой вопрос: **«на что похож этот товар — с точки зрения того, кто его покупал?»**

«Если пользователь любил "Матрицу" и "Бегущего по лезвию", а "Начало" исторически нравилось тем же людям, что и эти два фильма — порекомендуем "Начало"». Обратите внимание: мы не сравниваем пользователей друг с другом напрямую (как в Модуле 3) — мы сравниваем **товары** по тому, насколько похожим образом их оценивало **множество** пользователей.

### 4.1.2 Ключевой инсайт — почему это не просто «то же самое, только наоборот»

На первый взгляд может показаться, что IB-CF — это просто UB-CF с транспонированной матрицей, и разница чисто техническая. Это не так, и разница принципиальна:

- **В UB-CF** сходство считается между **пользователями** — сущностями, чьи предпочтения по определению изменчивы (Модуль 3.3.2).
- **В IB-CF** сходство считается между **товарами** — а отношения между товарами значительно более стабильны во времени. Тот факт, что «люди, любящие боевики Кристофера Нолана, также любят "Бегущего по лезвию"», — это устойчивая закономерность, которая не меняется от того, что конкретный пользователь второй месяц смотрит только комедии. Товар остаётся тем же товаром; связи между товарами меняются значительно медленнее, чем сиюминутные предпочтения отдельного человека.

Это единственная, но фундаментальная причина, по которой Item-Based подход исторически оказался настолько успешным в production — не потому что «сравнивать товары математически проще», а потому что **сама сравниваемая сущность стабильнее**.

## 4.2 Почему Item-Based обычно предпочтительнее User-Based в production

Здесь важно быть точным, а не повторять расхожее упрощение «у IB-CF сложность O(M²), у UB-CF — O(N²M), поэтому IB-CF быстрее». Это упрощение **не всегда корректно** — разберёмся детально.

### 4.2.1 Честный разбор вычислительной сложности

Расчёт полной матрицы сходства item-item в худшем случае стоит `O(M² × N)` — то есть **та же самая асимптотическая форма**, что и `O(N² × M)` у UB-CF (Модуль 3.3.1), только с переставленными местами `M` (число товаров) и `N` (число пользователей).

**Отсюда следует важный, часто упускаемый вывод:** IB-CF асимптотически дешевле UB-CF **только тогда, когда товаров меньше, чем пользователей** (`M < N`). Это верно для Netflix (десятки тысяч фильмов, десятки миллионов пользователей) — отсюда и успех подхода в этом кейсе. Но это **не универсально**: у крупного маркетплейса вроде Wildberries каталог может насчитывать десятки миллионов товаров, при том что активных пользователей — тоже десятки миллионов, а иногда товаров может быть даже **больше**, чем активных покупателей за период. В таком случае наивное сравнение "IB быстрее, потому что IB" — методологически некорректно, и решение нужно принимать исходя из конкретных `M` и `N` вашей системы, а не по умолчанию.

### 4.2.2 Настоящая причина предпочтения IB-CF — не сложность, а стабильность и предвычислимость

Ключевое практическое преимущество IB-CF — не в асимптотике, а в следующем: **отношения между товарами меняются медленно**, поэтому матрицу сходства товаров можно посчитать **один раз** (например, раз в сутки, batch-job) и переиспользовать для **всех** пользователей и **всех** запросов между пересчётами. У UB-CF аналогичный трюк работает намного хуже: даже если предвычислить сходство между всеми пользователями, эта матрица устареет намного быстрее (вкусы конкретных людей меняются быстрее, чем связи между товарами, — прямое следствие 4.1.2), и её пришлось бы пересчитывать значительно чаще, чтобы сохранить актуальность.

Именно это свойство — «посчитал редко, используешь часто» — делает IB-CF практичным для production **независимо** от того, `M < N` или `N < M` в конкретной системе. Это уточнение важно для честного ответа на собеседовании: если спросят «почему Item-Based лучше масштабируется», правильный ответ — не «потому что O(M²) < O(N²)» (это может быть неверно для конкретных чисел), а «потому что сходство между товарами стабильнее во времени и его можно эффективно кэшировать, что даёт O(1) на инференс вместо пересчёта на каждый запрос».

### 4.2.3 Сводная честная таблица

| Аспект | User-Based | Item-Based |
|:---|:---|:---|
| Асимптотика предвычисления | `O(N²M)` | `O(M²N)` — та же форма, просто переставлены роли |
| Что стабильнее | Вкусы людей изменчивы | Связи между товарами стабильнее |
| Возможность эффективного кэширования | Плохая (нужен частый пересчёт) | Хорошая (redко пересчитывается, часто используется) |
| Latency инференса | Требует пересчёта соседей на лету или частого обновления кэша | `O(1)` lookup в готовой предвычисленной матрице |
| Когда асимптотически выгоднее | Когда M ≫ N (товаров намного больше, чем пользователей) | Когда N ≫ M (пользователей намного больше, чем товаров) |
| Практический вывод | Реже используется как основной production-алгоритм | Чаще выбирается — но по причине стабильности/кэшируемости, а не только асимптотики |

## 4.3 Алгоритм

### 4.3.1 Adjusted Cosine Similarity — прямое применение Модуля 2.3.1

Вернёмся к тонкому моменту, уже отмеченному в Модуле 2.3.1: при сравнении **товаров** между собой всё равно нужно устранить смещение, вызванное «щедростью» **пользователей** (а не товаров) — потому что источник искажения именно в стиле оценивания конкретных людей. Поэтому используется **adjusted cosine similarity**:

In [ ]:
sim(i, j) = Σ_u∈U(r_ui - r̄_u)(r_uj - r̄_u) / (√Σ_u(r_ui - r̄_u)² × √Σ_u(r_uj - r̄_u)²)

где суммирование идёт по всем пользователям `U`, оценившим **оба** товара `i` и `j`, а `r̄_u` — средний рейтинг **пользователя** `u` (не товара!).

### 4.3.2 Формула предсказания

In [ ]:
pred(u, i) = Σ_j∈N(i)(sim(i,j) × r_uj) / Σ_j∈N(i)|sim(i,j)|

где `N(i)` — множество товаров, похожих на `i` (обычно top-k наиболее похожих), которые пользователь `u` уже оценил. Обратите внимание на отличие от формулы UB-CF (Модуль 3.2.2): здесь используются **сырые** оценки `r_uj` пользователя, а не их отклонения от среднего — потому что нормализация (центрирование по пользователю) уже была применена на этапе расчёта **сходства** (adjusted cosine, 4.3.1), а не на этапе финального предсказания.

### 4.3.3 Полный численный пример — та же задача, что в Модуле 3.2.3

Используем ровно ту же матрицу:

| | I1 | I2 | I3 | I4 | I5 |
|:---|:---:|:---:|:---:|:---:|:---:|
| U1 (целевой) | 5 | 3 | 4 | **?** | — |
| U2 | 4 | 2 | 3 | 4 | — |
| U3 | 1 | 5 | 2 | 1 | 5 |
| U4 | 5 | 3 | 5 | 5 | — |
| U5 | 2 | 2 | 1 | — | 3 |

**Задача та же:** предсказать оценку U1 для I4 — но теперь через сходство между **товарами**.

**Шаг 1. Центрируем всю матрицу по среднему каждого пользователя** (adjusted cosine, 4.3.1):

In [ ]:
r̄_U1 = (5+3+4)/3 = 4.0        r̄_U3 = (1+5+2+1+5)/5 = 2.8
r̄_U2 = (4+2+3+4)/4 = 3.25     r̄_U4 = (5+3+5+5)/4 = 4.5
                                r̄_U5 = (2+2+1+3)/4 = 2.0

Центрированная матрица (только заполненные ячейки):

| | I1 | I2 | I3 | I4 | I5 |
|:---|:---:|:---:|:---:|:---:|:---:|
| U1 | 1.0 | -1.0 | 0.0 | — | — |
| U2 | 0.75 | -1.25 | -0.25 | 0.75 | — |
| U3 | -1.8 | 2.2 | -0.8 | -1.8 | 2.2 |
| U4 | 0.5 | -1.5 | 0.5 | 0.5 | — |
| U5 | 0.0 | 0.0 | -1.0 | — | 1.0 |

**Шаг 2. Считаем adjusted cosine между I4 и каждым товаром, который оценил U1 (I1, I2, I3)**, используя только пользователей, оценивших **оба** сравниваемых товара (U2, U3, U4 — единственные, кто оценил I4).

*sim(I4, I1):*

In [ ]:
I4 (у U2,U3,U4): [0.75, -1.8, 0.5]
I1 (у U2,U3,U4): [0.75, -1.8, 0.5]

Векторы идентичны (потому что в исходных данных I1 и I4 получили от U2, U3, U4 буквально одинаковые оценки) ->
sim(I4, I1) = 1.0

*sim(I4, I2):*

In [ ]:
I4: [0.75, -1.8, 0.5]      I2 (у U2,U3,U4): [-1.25, 2.2, -1.5]

числитель = 0.75×(-1.25) + (-1.8)×2.2 + 0.5×(-1.5) = -0.9375 - 3.96 - 0.75 = -5.6475
||I4|| = √(0.75² + 1.8² + 0.5²) = √4.0525 ≈ 2.013
||I2|| = √(1.25² + 2.2² + 1.5²) = √8.6525 ≈ 2.942

sim(I4, I2) = -5.6475 / (2.013 × 2.942) ≈ -0.954

*sim(I4, I3):*

In [ ]:
I4: [0.75, -1.8, 0.5]      I3 (у U2,U3,U4): [-0.25, -0.8, 0.5]

числитель = 0.75×(-0.25) + (-1.8)×(-0.8) + 0.5×0.5 = -0.1875 + 1.44 + 0.25 = 1.5025
||I3|| = √(0.25² + 0.8² + 0.5²) = √0.9525 ≈ 0.976

sim(I4, I3) = 1.5025 / (2.013 × 0.976) ≈ 0.765

**Шаг 3. Предсказание с использованием всех трёх соседних товаров (включая отрицательный):**

In [ ]:
числитель = sim(I4,I1)×r_{U1,I1} + sim(I4,I2)×r_{U1,I2} + sim(I4,I3)×r_{U1,I3}
          = 1.0×5 + (-0.954)×3 + 0.765×4
          = 5.0 - 2.862 + 3.06 = 5.198

знаменатель = |1.0| + |-0.954| + |0.765| = 2.719

pred(U1, I4) = 5.198 / 2.719 ≈ 1.911

### 4.3.4 Критически важное наблюдение — почему этот результат подозрителен

Результат `1.911` выглядит крайне неправдоподобно: U1 поставил высокую оценку товару I1, который **идеально коррелирует** с I4 (`sim = 1.0`), а итоговое предсказание получилось **низким**. Разберём, почему так вышло.

**Причина:** в формуле предсказания IB-CF (4.3.2) используются **сырые** оценки `r_uj`, а не их отклонения, но при этом **сходство может быть отрицательным**. Когда отрицательное сходство `sim(I4, I2) = -0.954` умножается на **сырую положительную** оценку `r_{U1,I2} = 3`, получается крупный отрицательный вклад в числитель — который не имеет такой же ясной, «отрицание-компенсирует-отрицание» интерпретации, как в формуле UB-CF (сравните с Модулем 3.2.4, где отрицательное сходство, умноженное на отрицательное **отклонение**, давало осмысленный положительный вклад). Здесь же отрицательное сходство просто «вычитает» часть сырой оценки, что не имеет прямого содержательного смысла — оценка `3` сама по себе не была ни выше, ни ниже среднего в каком-то однозначном отношении к I2.

**Практическое решение, повсеместно применяемое в реальных реализациях IB-CF:** использовать в предсказании **только положительно коррелирующие** соседние товары, отбрасывая отрицательные сходства **на этапе финального предсказания** (хотя они всё ещё могут быть полезны, например, для функции «анти-рекомендаций» — «если любите I2, вряд ли вам понравится I4»).

**Пересчитаем предсказание, используя только I1 и I3 (положительные сходства):**

In [ ]:
числитель = 1.0×5 + 0.765×4 = 5.0 + 3.06 = 8.06
знаменатель = 1.0 + 0.765 = 1.765

pred(U1, I4) = 8.06 / 1.765 ≈ 4.566

**Это заметно более правдоподобный результат** — согласуется по порядку величины с предсказанием UB-CF из Модуля 3 (`5.0` после клиппинга). Оба алгоритма сходятся во мнении, что U1, вероятно, высоко оценит I4, хотя точные числа отличаются — это нормально: два разных алгоритма, использующих разные допущения, редко дают идентичные предсказания, и на практике их часто комбинируют именно поэтому (прямая мотивация Модуля 7 — гибридные системы).

**Вывод, который стоит явно унести с этого раздела:** отрицательное сходство в UB-CF (Модуль 3) и в IB-CF (этот раздел) **интерпретируется и обрабатывается по-разному** из-за разницы в том, что именно взвешивается — отклонение (UB-CF) или сырая оценка (IB-CF). Это тонкий, но важный нюанс, который часто упускают, и хороший источник вопросов на техническом интервью.

## 4.4 Предвычисление и кэширование

### 4.4.1 Архитектура batch-пересчёта

В отличие от UB-CF, где пересчёт сходства «на лету» — почти неизбежное зло, IB-CF органично ложится на следующую архитектуру:

1. **Batch job** (например, раз в сутки, ночью, в период низкой нагрузки) пересчитывает полную (или top-k) матрицу сходства товаров по всем накопленным на текущий момент данным.
2. Результат сохраняется в быстро читаемое хранилище (Redis, отдельная таблица в PostgreSQL — прямая аналогия с `item_embeddings` из вашего проекта FraudGuard/RecoStage, Модуль 13.3).
3. В момент реального запроса (инференс) нужен только **lookup** — `O(1)` на пару товаров вместо полного пересчёта.

### 4.4.2 Проблема хранения при очень большом каталоге

Если товаров `M` относительно немного (тысячи-десятки тысяч, как в кино/музыке), полную матрицу `M × M` можно хранить целиком:

In [ ]:
10 000 × 10 000 × 4 байта (float32) ≈ 400 000 000 байт ≈ 381 МБ — вполне приемлемо.

Но если `M` — миллионы (крупный e-commerce каталог), полная матрица `M²` становится нереализуемой:

In [ ]:
10 000 000² × 4 байта = 4 × 10^14 байт — абсолютно нереализуемо.

**Практическое решение:** хранить не полную матрицу, а только **top-k наиболее похожих товаров для каждого товара** (например, top-50). Это резко сокращает объём хранения:

In [ ]:
10 000 000 товаров × 50 соседей × (4 байта id + 4 байта similarity) = 10 000 000 × 50 × 8 = 4 000 000 000 байт ≈ 3.7 ГБ

— уже вполне управляемый объём, легко умещающийся в оперативную память или быстрое key-value хранилище (Redis).

### 4.4.3 Код предвычисления и кэширования top-k соседей

In [ ]:
import numpy as np
import scipy.sparse as sp

def compute_item_similarity_topk(ratings_matrix: sp.csr_matrix, k: int = 50) -> dict:
    """
    Предвычисление top-k похожих товаров для каждого товара.
    ratings_matrix - разреженная матрица (Модуль 2.1), уже центрированная по строкам (adjusted cosine).
    """
    # Транспонируем для работы с товарами как со строками (используем CSC->CSR трюк, Модуль 2.1.4)
    item_user_matrix = ratings_matrix.T.tocsr()

    # Нормализация строк (для последующего скалярного произведения = косинус)
    norms = sp.linalg.norm(item_user_matrix, axis=1)
    norms[norms == 0] = 1e-10
    normalized = item_user_matrix.multiply(1 / norms[:, None]).tocsr()

    # Полная матрица сходства (для очень больших M нужно считать блоками, не целиком)
    similarity_matrix = normalized @ normalized.T

    top_k_neighbors = {}
    for item_id in range(similarity_matrix.shape[0]):
        row = similarity_matrix.getrow(item_id).toarray().flatten()
        row[item_id] = -np.inf  # исключаем сходство товара с самим собой
        top_k_idx = np.argpartition(-row, k)[:k]
        top_k_neighbors[item_id] = [
            (int(idx), float(row[idx])) for idx in top_k_idx if row[idx] > 0  # только положительные (4.3.4)
        ]

    return top_k_neighbors

# Сохранение в Redis (batch job, обновляется раз в сутки — 4.4.1)
import json
import redis

r = redis.Redis(host='localhost', port=6379, db=0)

def cache_item_similarities(top_k_neighbors: dict):
    for item_id, neighbors in top_k_neighbors.items():
        r.set(f"item_sim:{item_id}", json.dumps(neighbors), ex=86400 * 2)  # TTL 2 суток

## 4.5 Практика

### 4.5.1 Полная реализация adjusted cosine и предсказания — воспроизведение примера из 4.3.3

In [ ]:
import numpy as np
import pandas as pd

ratings = pd.DataFrame({
    'I1': [5, 4, 1, 5, 2],
    'I2': [3, 2, 5, 3, 2],
    'I3': [4, 3, 2, 5, 1],
    'I4': [np.nan, 4, 1, 5, np.nan],
    'I5': [np.nan, np.nan, 5, np.nan, 3],
}, index=['U1', 'U2', 'U3', 'U4', 'U5'])

def adjusted_cosine(ratings: pd.DataFrame, item_i: str, item_j: str) -> float:
    """Adjusted cosine similarity между двумя товарами (Модуль 4.3.1)."""
    user_means = ratings.mean(axis=1)
    common_users = ratings[item_i].notna() & ratings[item_j].notna()
    if common_users.sum() < 2:
        return 0.0

    i_centered = ratings.loc[common_users, item_i] - user_means[common_users]
    j_centered = ratings.loc[common_users, item_j] - user_means[common_users]

    denom = np.linalg.norm(i_centered) * np.linalg.norm(j_centered)
    if denom == 0:
        return 0.0
    return float(np.dot(i_centered, j_centered) / denom)

def predict_rating_itembased(ratings: pd.DataFrame, target_user: str, item: str,
                               positive_only: bool = True) -> float:
    """Предсказание через Item-Based CF (Модуль 4.3.2)."""
    user_rated_items = ratings.loc[target_user].dropna().index.tolist()

    similarities = {}
    for other_item in user_rated_items:
        sim = adjusted_cosine(ratings, item, other_item)
        if positive_only and sim <= 0:
            continue  # отбрасываем отрицательные (4.3.4)
        if sim != 0:
            similarities[other_item] = sim

    if not similarities:
        return float(ratings[item].mean())  # fallback: item-mean (холодный старт, 1.5.1)

    numerator = sum(sim * ratings.loc[target_user, j] for j, sim in similarities.items())
    denominator = sum(abs(sim) for sim in similarities.values())

    return float(np.clip(numerator / denominator, 1, 5))

pred_with_negative = predict_rating_itembased(ratings, 'U1', 'I4', positive_only=False)
pred_positive_only = predict_rating_itembased(ratings, 'U1', 'I4', positive_only=True)

print(f"С учётом отрицательных сходств: {pred_with_negative:.3f}")   # ≈ 1.911
print(f"Только положительные сходства: {pred_positive_only:.3f}")     # ≈ 4.566

### 4.5.2 Сравнение скорости: UB-CF на лету vs IB-CF с кэшем

In [ ]:
import time

# Симуляция "холодного" пересчёта UB-CF на каждый запрос (Модуль 3, без кэша)
start = time.time()
for _ in range(100):  # 100 "запросов"
    _ = predict_rating(ratings, 'U1', 'I4', k=5)  # функция из Модуля 3.4.1
ub_cf_time = time.time() - start

# Симуляция IB-CF с предвычисленным кэшем сходств (реалистичный сценарий production)
precomputed_sims = {
    ('I4', 'I1'): 1.0, ('I4', 'I2'): -0.954, ('I4', 'I3'): 0.765
}

def predict_from_cache(target_user, item, ratings, precomputed_sims):
    user_rated = ratings.loc[target_user].dropna().index.tolist()
    sims = {j: precomputed_sims.get((item, j), 0) for j in user_rated if precomputed_sims.get((item, j), 0) > 0}
    if not sims:
        return ratings[item].mean()
    num = sum(s * ratings.loc[target_user, j] for j, s in sims.items())
    den = sum(abs(s) for s in sims.values())
    return num / den

start = time.time()
for _ in range(100):
    _ = predict_from_cache('U1', 'I4', ratings, precomputed_sims)
ib_cf_cached_time = time.time() - start

print(f"UB-CF (пересчёт каждый раз): {ub_cf_time:.4f} сек")
print(f"IB-CF (из кэша): {ib_cf_cached_time:.4f} сек")
# На реальных данных разница будет измеряться порядками, не процентами

### 4.5.3 Полномасштабная практика на MovieLens

- Предвычислить полную матрицу adjusted cosine similarity между всеми парами фильмов (на подвыборке, аналогично Модулю 3.4.3).
- Реализовать top-k кэш (аналогично 4.4.3) и сравнить объём занимаемой памяти: полная матрица vs top-50 на товар (прямое воспроизведение расчёта из 4.4.2 на реальных числах вашей подвыборки).
- Посчитать RMSE предсказаний IB-CF на temporal split (Модуль 1.7) — с учётом отрицательных сходств и без — и сравнить с результатом UB-CF из Модуля 3.4.3. Обсудить (письменно, для себя): согласуются ли алгоритмы в целом, и если нет — на каких именно пользователях/товарах расхождение больше всего (обычно — там, где данных меньше всего, что прямая связь с проблемой значимости сходства, Модуль 3.3.4).

### 4.5.4 Вопросы для самопроверки

1. Почему заявление «Item-Based CF асимптотически быстрее User-Based CF, потому что M < N» некорректно как универсальное утверждение? При каких условиях оно верно, а при каких — нет?
2. В чём разница между тем, как отрицательное сходство обрабатывается в формуле предсказания UB-CF (Модуль 3.2.2) и в формуле IB-CF (4.3.2)? Почему в одной формуле оно осмысленно, а в другой — нет без дополнительной обработки?
3. Почему для каталога с миллионами товаров нельзя хранить полную item-item матрицу сходства, и какое практическое решение применяется вместо этого?
4. Мы получили два разных предсказания для одной и той же пары (пользователь, товар) — 4.566 (IB-CF, только положительные соседи) и 5.0 (UB-CF, с клиппингом). Если бы вам нужно было выбрать одно финальное число для показа пользователю, как бы вы поступили? (Подсказка: это прямой практический вопрос, отвечающий на который, вы придёте к идее Модуля 7 — гибридных систем.)

## Глоссарий модуля 4

| Термин | Короткое определение |
|:---|:---|
| Item-Based CF | Рекомендация на основе похожих товаров, а не похожих пользователей |
| Adjusted Cosine Similarity | Косинус между товарами с центрированием оценок по среднему пользователя |
| Item-to-Item Collaborative Filtering | Историческое название подхода Amazon (Linden, Smith, York, 2003) |
| Top-k neighbor caching | Хранение только k наиболее похожих соседей вместо полной матрицы сходства |
| Batch-пересчёт | Периодическое (не в реальном времени) обновление предвычисленных сходств |
| Positive-only filtering | Практика отбрасывать отрицательные сходства при финальном предсказании IB-CF |

**Связь со следующим модулем:** оба алгоритма, разобранные в Модулях 3 и 4, — **memory-based**: они не строят компактную обученную модель, а каждый раз (или через кэш) обращаются к сырым данным напрямую. Модуль 5 совершает переход к принципиально другому классу методов — **model-based**, где вместо хранения и сравнения сырых векторов оценок обучается компактное скрытое представление (эмбеддинг) каждого пользователя и товара, что одновременно решает проблему масштабируемости (3.3.1, 4.2.1) и проблему разреженности (Модуль 1.4.2) за один шаг.